# Aplicações da Inteligência Artificial

Prof. [Denilson Alves Pereira](https://sites.google.com/ufla.br/denilsonpereira) <br>
Departamento de Ciência da Computação (DCC) <br>
Instituto de Ciências Exatas e Tecnológicas (ICET) <br> 
Universidade Federal de Lavras (UFLA)

## RAG - Retrieval-Augmented Generation (Geração Aumentada de Recuperação)

Objetivo: apresentar o conceito de RAG e um exemplo de implementação.

Você vai aprender a:
- Aplicar os conceitos de RAG em uma implementação básica
- Usar um modelo de linguagem para codificar, na forma de embeddings, uma base de conhecimento textual
- Usar um banco de dados vetorial para armazenar os embeddings da base de conhecimento
- Gerar o contexto para responder consultas do usuário a partir dos embeddings da base de conhecimento
- Usar um modelo de linguagem pré-treinados para responder as consultas do usuário

Referências: <br>
https://medium.com/@danushidk507/rag-with-llama-using-ollama-a-deep-dive-into-retrieval-augmented-generation-c58b9a1cfcd3
https://huggingface.co/blog/ngxson/make-your-own-rag <br>
https://aws.amazon.com/pt/what-is/retrieval-augmented-generation/

Versão: Janeiro, 2025

## Conceito de RAG

Os Grandes Modelos de Linguagem (LLM - *Large Language Model*) são treinados em grandes volumes de dados e podem ser usados para tarefas como responder perguntas e traduzir idiomas. Entretanto, eles são genéricos, e podem não ter conhecimentos detalhados em domínios específicos ou da base de dados da sua empresa.

Retrieval-Augmented Generation (RAG) é o processo de usar um LLM para responder consultas de usuários fazendo referência a uma base de conhecimento fora das fontes de dados em que ele foi treinado. RAG combina recuperação de informação com geração de texto. Dessa forma, é possível responder perguntas acessando informações atualizadas ou de domínio específico.

<img src="./figs/rag-fluxograma.jpg" width="720" height="400" alt="RAG Fluxograma"> <br>
Referência: https://aws.amazon.com/pt/what-is/retrieval-augmented-generation/

## Componentes de um Sistema de RAG

Um sistema de RAG consiste de dois componentes principais:
- Um modelo de recuperação que busca informação relevante de uma fonte de conhecimento externo, a qual pode ser um banco de dados, uma máquina de busca ou qualquer outro repositório de informação.
- Um modelo de linguagem (LLM) que gera respostas baseadas no conhecimento recuperado.

## RAG Simples

Vamos criar um sistema de RAG simples que recupera informação de um dataset predefinido e gera respostas baseadas no conhecimento recuperado. O sistema compreenderá os seguintes componentes:
1. Modelo de Embedding: um modelo de linguagem predefinido que converte textos de entrada em embeddings (representação vetorial que captura o significado semântico). Esses vetores serão usados para pesquisar informação relevante no dataset.
2. Banco de Dados Vetorial: Um sistema de armazenamento para a base de conhecimento e seus correspondentes vetores de embedding. Existem várias tecnologias para bancos de dados vetoriais, tais como FAISS, pgvector, Qdrant e Pinecone.
3. Chatbot: Um modelo de linguagem que gera respostas baseadas no conhecimento recuperado. Esse pode ser qualquer modelo de linguagem, tais como Llama, Gemma ou GPT.

<img src="./figs/rag-simples.svg" width="240" height="140" alt="RAG simples"> <br>
Referência: https://huggingface.co/blog/ngxson/make-your-own-rag

## Fase de Indexação

A indexação é o primeiro passo na criação de um sistema de RAG. Ela envolve quebrar os documentos em blocos (*chunks*) menores e calcular a representação vetorial de cada bloco, de forma que possam ser eficientemente buscados durante a geração.

<img src="./figs/rag-indexacao.svg" width="400" height="240" alt="RAG Indexação"> <br>
Referência: https://huggingface.co/blog/ngxson/make-your-own-rag

O tamanho de cada bloco pode variar dependendo da base de dados e da aplicação. Por exemplo, em um sistema de recuperação de documentos, cada bloco pode ser um parágrafo ou uma sentença. Em um sistema de diálogo, cada bloco pode ser o trecho da mudança da conversa.

Depois da fase de indexação, cada bloco com seu correspondente vetor de embedding será armazenado no banco de dados vetorial.

## Fase de Recuperação

Dada como entrada a consulta do usuário, o sistema calcula a representação vetorial dessa consulta e a compara com os vetores no banco de dados para encontrar os blocos mais relevantes. Para comparar a similaridade entre os vetores de embeddings, consulta e cada bloco armazenado, nós podemos usar a Similaridade do Cosseno, a Distância Euclidiana, ou outras métricas de distância.

O resultado retornado pelo banco de dados vetorial conterá os top N blocos mais relevantes para a consulta. Esses blocos serão usados pelo Chatbot para gerar a resposta.

<img src="./figs/rag-recuperacao.svg" width="300" height="180" alt="RAG Recuperação"> <br>
Referência: https://huggingface.co/blog/ngxson/make-your-own-rag

## Fase de Geração

Os blocos relevantes recuperados na fase de recuperação são usados para alimentar o LLM com o contexto para gerar respostas coerentes, baseadas nos fatos da base de conhecimento. Para isso, o LLM usa tanto o contexto quanto seu conhecimento interno.

#### Fluxograma das três fases:

<img src="./figs/rag-fluxograma.webp" width="600" height="480" alt="RAG Fluxograma"> <br>
Referência: https://medium.com/@danushidk507/rag-with-llama-using-ollama-a-deep-dive-into-retrieval-augmented-generation-c58b9a1cfcd3

## Código em Python para um Sistema de RAG

#### Ollama

Faça o download da ferramenta Ollama. Ela é um framework que permite que você execute modelos diretamente no seu computador, sem a necessidade de acessar um servidor ou serviço em nuvem.

Instale o Ollama a partir do website do projeto: https://ollama.com/

Depois, abra um terminal e execute o seguinte comando para fazer o download do modelo requerido: <br>
ollama pull hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF

Finalmente, para usar o Ollama em Python, instale o pacote: <br>
pip install ollama

#### Llama

Para o LLM do chatbot, vamos usar o Llama, um modelo desenvolvido pela Meta (Facebook). Ele foi liberado sob uma licença 'open access', e pode ser usado localmente em seu servidor, sem a necessidade de pagar o acesso via uma API. LlaMa está disponível em vários tamanhos (quantidade de parâmetros), permitindo o desenvolvedor escolher o modelo baseado em seu caso de uso e restrições computacionais.

#### LangChain

LangChain é um framework para se construir aplicações usando LLMs e ferramentas externas. Ele pode ser usado para orquestrar o fluxograma do RAG.

#### Ambiente

Crie um ambiente para o projeto. <br>
Execute o código abaixo em um terminal do sistema operacional.

In [1]:
# Baixe o Ollama de https://ollama.com/

# Baixe o modelo a ser usado no exemplo:
# ollama pull hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF

# Crie um ambiente para o projeto
# python3 -m venv $HOME/rag-env
# source $HOME/rag-env/bin/activate
# pip install jupyter

# Instale os seguinte pacotes
# pip install ollama
# pip install langchain
# pip install -U langchain-community
# pip install sentence-transformers
# pip install faiss-gpu
# pip install pypdf
# pip install langchain_ollama
# pip install colab-xterm

### Base de Conhecimento

Para este exemplo, vamos usar o regulamento de cursos da Universidade Federal de Lavras (UFLA) como a base de conhecimento a ser consultada. Ele é um documento de 138 páginas, no formato PDF.

As classes PyPDFLoader e CharacterTextSplitter serão usadas para carregar o documento e dividi-lo em blocos (chunks), sobrepondo parte dos blocos para uma melhor acurácia da recuperação.

In [2]:
# Data Ingestion (carga e indexação dos dados)
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter

# Carrega o documento
loader = PyPDFLoader("./datasets/resolucao-cepe-473-2018-regulamento-cursos-grad.pdf")
documents = loader.load()

# Divide o documento em chunks separados por fim-de-lina, com tamanho máximo de 1.000 caracteres e com sobreposição de 30 caracteres.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30, separator="\n")
docs = text_splitter.split_documents(documents=documents)

### Banco de Dados Vetorial

Para implementação do banco de dados vetorial, vamos utilizar o FAISS (Facebook AI Similarity Search). Ele é uma biblioteca versátil e eficiente para busca por similaridade vetorial.

Para o modelo de embedding, vamos utilizar um modelo em português disponibilizado na plataforma HuggingFace.

In [3]:
# Modelos
EMBEDDING_MODEL = 'ricardo-filho/bert-base-portuguese-cased-nli-assin-2'
#EMBEDDING_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF'

In [4]:
# Data Embedding and Storage with FAISS

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# Load embedding model
model_kwargs = {"device": "cuda"}
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs=model_kwargs
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(docs, embeddings)

# Save and reload the vector store
vectorstore.save_local("faiss_index_")
persisted_vectorstore = FAISS.load_local("faiss_index_", embeddings, allow_dangerous_deserialization=True)

# Create a retriever
retriever = persisted_vectorstore.as_retriever()

/tmp/ipykernel_7279/2056980109.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


### Fases de Recuperação e de Geração

O chatbot vai gerar a resposta baseada no conhecimento recuperado do banco de dados vetorial a partir da consulta. Isso é feito simplesmente adicionando-se os chunks dentro do prompt que será levado como entrada para o chatbot.

No LangChain, a orquestração do RAG é feita com a classe RetrievalQA. Nós combinamos o retriever e o Llama dentro de uma cadeia RetrievalQA. Isso permite perguntas e respostas interativas baseadas nos documentos ingeridos.

In [5]:
# Carregando o LlamA com Ollama

# Para rodar no Google Colab
#%xterm
#curl -fsSL https://ollama.com/install.sh | sh
#ollama serve & ollama pull llama3.1

from langchain_community.llms import Ollama

# Inicializa o modelo Llama
llm = Ollama(model=LANGUAGE_MODEL)

# Testa com um prompt simples
response = llm.invoke("Conte-me uma piada")
print(response)

/tmp/ipykernel_7279/1120485283.py:11: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model=LANGUAGE_MODEL)


Qual é a razão pela qual o coelho não consegue parar de correr? Porque ele tem muita pressa.


In [6]:
# Orquestrando o RAG com RetrievalQA
from langchain.chains import RetrievalQA

# Cria RetrievalQA
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

# Interage em um loop de consultas
while True:
    query = input("Digite sua consulta (ou tecle 'Sair' para sair): \n")
    if query.lower() == "sair":
        break
    result = qa.run(query)
    print(result)

Digite sua consulta (ou tecle 'Sair' para sair): 
 O que é a Área Básica de Ingresso (ABI)?


/tmp/ipykernel_7279/1502486393.py:12: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa.run(query)


A Área Básica de Ingresso (ABI) é um tipo de organização que congrega 2 (dois) ou mais cursos da mesma área de formação em um único ingresso, por processo seletivo.


Digite sua consulta (ou tecle 'Sair' para sair): 
 sair


## Comentários Finais

Esse notebook apresentou os conceitos básicos de RAG e uma implementação simples. Há ainda muito espaço para melhorias, tais como quebrar consultas complexas em tópicos, re-ranquear os chunks recuperados, usar técnicas mais sofisticadas para quebrar os documentos em chunks, usar um modelo de linguagem com maior capacidade (maior número de parâmetros). Além disso, pode-se representar o conhecimento por meio de grafos (Graph RAG). RAG é um tópico de pesquisa em constante evolução. Para se obter bons resultados, é preciso estar atento ao estado-da-arte na pesquisa.